# Emparejamiento bipartito máximo para cobertura auditable

**Proyecto educativo de algoritmos — maestría en ingeniería estadística.**  
Este notebook no diagnostica, recomienda tratamientos, prioriza pacientes ni procesa historias clínicas. Los objetos son escenarios simbólicos y obligaciones de cobertura de una fuente estructurada.

**Goal:** construir y verificar una asignación uno-a-uno máxima, trazable y reproducible.

Ruta: `INSPECT → PLAN → IMPLEMENT → VERIFY → EVIDENCE AUDIT → REVIEW → RECORD`.

## 1. Preparar un entorno limpio

En Colab se clona la rama del proyecto. En ejecución local se usa el checkout actual. NetworkX se instala únicamente para los oráculos.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/llabans/grafos-LLM.git'
BRANCH = 'feat/bipartite-matching-minsa'

if IN_COLAB:
    project_root = Path('/content/grafos-LLM')
    if not project_root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(project_root)], check=True)
else:
    project_root = Path.cwd()
    if not (project_root / 'pyproject.toml').exists():
        project_root = project_root.parent

os.chdir(project_root)
if IN_COLAB and shutil.which('dot') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'graphviz'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]', 'graphviz', 'pandas'], check=True)
print('Proyecto:', project_root)

## 2. Proporcionar la fuente local

El ZIP no se versiona. En Colab, suba `integration_design_data(1).zip`. En local, defina `BIPARTITE_MINSA_SOURCE_ROOT`. La celda busca la carpeta que contiene `snapshots/gpc_json/gpc-05-hipertension-embarazo-2025`.

In [ ]:
import zipfile

configured = os.environ.get('BIPARTITE_MINSA_SOURCE_ROOT')
if configured:
    source_root = Path(configured).expanduser().resolve()
elif IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
    extract_root = Path('/content/gpc_source')
    extract_root.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall(extract_root)
    marker = next(extract_root.rglob('snapshots/gpc_json/gpc-05-hipertension-embarazo-2025'))
    source_root = marker.parents[2]
else:
    raise RuntimeError('Defina BIPARTITE_MINSA_SOURCE_ROOT antes de continuar')

os.environ['BIPARTITE_MINSA_SOURCE_ROOT'] = str(source_root)
print('Fuente configurada:', source_root)

## 3. Gate de fuente: hashes, alcance y conteos

Un nombre de archivo no prueba identidad. Se validan ocho SHA-256 y luego se filtra Phase 4 por `source_id` exacto.

In [ ]:
subprocess.run([sys.executable, 'scripts/validate_sources.py', '--source-root', str(source_root), '--output', 'reports/runtime/colab_source_validation.json'], check=True)

In [ ]:
import pandas as pd
from bipartite_minsa.pipeline.ingest import ingest_sources

ingestion = ingest_sources(source_root)
pd.DataFrame([
    {'métrica': key, 'observado': value, 'esperado': ingestion.source_manifest['baseline_counts'][key]}
    for key, value in ingestion.source_manifest['observed_counts'].items()
])

## 4. Primera capa: grafo dirigido `D=(N,A)`

`D` conserva el flujo. La reconciliación primaria/derivada debe ser exacta; `flow_2:n9` permanece como aislado informativo.

In [ ]:
pd.DataFrame([
    {
        'flujo': graph.graph_id.rsplit('__flow__', 1)[-1],
        'nodos': len(graph.nodes),
        'aristas': len(graph.transitions),
        'alcanzables': len(graph.reachable_node_ids),
        'alcance_clínico-estructural': graph.clinical_scope,
        'reconciliado': report.is_equivalent,
    }
    for graph, report in zip(ingestion.graphs, ingestion.reconciliations)
])

In [ ]:
from graphviz import Source
from bipartite_minsa.visualization import directed_graph_dot

flow_2 = next(graph for graph in ingestion.graphs if graph.graph_id.endswith('__flow__flow_2'))
Source(directed_graph_dot(flow_2))

## 5. Firmas, escenarios `U` y objetivos `V`

Los escenarios no son pacientes. Se generan por reglas combinatorias versionadas. `10 + C(11,2) + 1 + 1 + 1 + 7 = 75`.

In [ ]:
from collections import Counter
from bipartite_minsa.pipeline.integrate import integrate_sources

integration = integrate_sources(ingestion)
print('Firmas:', len(integration.signatures), '|U|:', len(integration.scenarios), '|V|:', len(integration.targets))
pd.DataFrame(Counter(s.generation_rule_id for s in integration.scenarios).items(), columns=['regla', 'escenarios']).sort_values('regla')

In [ ]:
pd.DataFrame([
    {
        'scenario_id': item.scenario_template_id,
        'predicados': ', '.join(item.input_predicates),
        'regla': item.generation_rule_id,
        'fuentes': len(item.source_refs),
    }
    for item in integration.scenarios[:8]
])

## 6. Segunda capa: compatibilidad y grafo `B=(U,V,E_B)`

Una arista exige predicados, acciones, estado, guía y procedencia. No se usan similitudes ni conocimiento externo.

In [ ]:
from bipartite_minsa.pipeline.build_bipartite_graph import build_graph

built = build_graph(integration)
decision_counts = Counter(d.status.value for d in built.decisions)
print('|U|=', len(built.graph.left), '|V|=', len(built.graph.right), '|E|=', built.graph.m)
pd.Series(decision_counts, name='pares').to_frame()

## 7. Camino aumentante en un fixture matemático

En `u2-v1-u1-v2`, las aristas alternan fuera/dentro/fuera de `M`. Invertirlas aumenta la cardinalidad de 1 a 2.

In [ ]:
from bipartite_minsa.matching.graph import BipartiteGraph
from bipartite_minsa.matching.augmenting_path import AugmentingPathMatcher

toy = BipartiteGraph(('u1','u2'), ('v1','v2'), (('u1','v1'),('u1','v2'),('u2','v1')))
toy_result = AugmentingPathMatcher().solve(toy)
print('Matching máximo:', toy_result.pairs, 'cardinalidad=', toy_result.cardinality)

## 8. DFS frente a Hopcroft–Karp

DFS aumenta un camino por búsqueda. Hopcroft–Karp usa BFS por capas y DFS admisible para aumentar varios caminos mínimos por fase.

In [ ]:
from bipartite_minsa.matching.hopcroft_karp import HopcroftKarpMatcher
from bipartite_minsa.matching.oracles import networkx_cardinality

dfs_result = AugmentingPathMatcher().solve(built.graph)
hk_result = HopcroftKarpMatcher().solve(built.graph)
oracle = networkx_cardinality(built.graph)
pd.DataFrame([
    {'algoritmo': 'DFS propio', 'cardinalidad': dfs_result.cardinality, 'escaneos': dfs_result.instrumentation.edge_scans, 'fases BFS': 0},
    {'algoritmo': 'HK propio', 'cardinalidad': hk_result.cardinality, 'escaneos': hk_result.instrumentation.edge_scans, 'fases BFS': hk_result.instrumentation.bfs_phases},
    {'algoritmo': 'NetworkX (oráculo)', 'cardinalidad': oracle, 'escaneos': None, 'fases BFS': None},
])

Complejidad: DFS `O(nm)`; Hopcroft–Karp `O(m√n)` sobre el grafo activo y `O(n+m√n)` al contar aislados explícitos; espacio `O(n+m)`.

## 9. Resultado, métricas y testigos de Hall

El porcentaje es cobertura combinatoria de objetivos, no accuracy ni cobertura de pacientes.

In [ ]:
from bipartite_minsa.metrics import graph_metrics
from bipartite_minsa.matching.analysis import hall_deficiency_witnesses

metrics = graph_metrics(built.graph, hk_result)
pd.Series({k: v for k, v in metrics.items() if k not in {'unused_scenarios','uncovered_targets','left_degree_distribution','right_degree_distribution'}}, name='valor').to_frame()

In [ ]:
witnesses = hall_deficiency_witnesses(built.graph, hk_result)
pd.DataFrame([
    {
        'mecanismo': w.mechanism,
        '|S| objetivos': len(w.target_ids),
        '|N(S)| escenarios': len(w.neighboring_scenario_ids),
        'déficit': w.deficiency,
        'raíz libre': w.root_unmatched_target,
    }
    for w in witnesses
])

Los déficits `1>0` y `8>7` implican `|M|≤35−2=33`. Como se construye un matching de 33, el óptimo queda certificado.

## 10. Pruebas exhaustivas y oráculos

La suite recorre los 689 grafos etiquetados con particiones de tamaño 0 a 3 y compara DFS, HK, enumeración exhaustiva y NetworkX.

In [ ]:
test_env = dict(os.environ)
test_env['BIPARTITE_MINSA_SOURCE_ROOT'] = str(source_root)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, env=test_env)

## 11. Teoría → código → prueba


In [ ]:
pd.DataFrame([
    ['Camino aumentante', 'CLRS §25.1, imp. 705-708 / PDF 913-917', 'matching/augmenting_path.py', 'casos + exhaustivo'],
    ['BFS por capas', 'CLRS §25.1, imp. 709-711 / PDF 918-921', 'matching/hopcroft_karp.py', 'fases e invariantes'],
    ['DFS admisible', 'CLRS §25.1, imp. 712 / PDF 922', 'matching/hopcroft_karp.py', 'paridad con oráculos'],
    ['O(m√n)', 'CLRS Teo. 25.8, imp. 715 / PDF 926', 'experiments.py', 'benchmark controlado'],
    ['Déficit de Hall', 'CLRS imp. 715 / PDF 926-927', 'matching/analysis.py', 'testigos 1>0 y 8>7'],
], columns=['concepto','fuente','código','verificación'])

## 12. Conclusión y límites

- Resultado: `75, 35, 267, 33`, cobertura uno-a-uno `94,3%`.
- Correctitud: dos solvers propios, enumeración, NetworkX y Hall.
- Trazabilidad: archivo, entidad, página/sección y SHA-256.
- Límite: una arista ausente significa evidencia estructural insuficiente bajo esta regla; no juicio clínico.

**Máxima cobertura auditable. Cero decisiones clínicas.**